# Notebook 3: Agent Fundamentals

**🎯 Goal:** Understand what an Agent is, its key components, and how to use a pre-built agent with simple tools.

## 🧩 Concept Overview

An LLM-powered agent is not just a chatbot — it’s a system that can reason, plan, and act to accomplish a task. It operates in a loop:

1.  **Observes** the user's input.
2.  **Thinks** about what step to take next.
3.  **Acts** by using a tool.
4.  **Observes** the result of the action.
5.  **Repeats** this process until the task is complete.

Think of it as an AI that **thinks and does**, not just answers.

### Key Components
- **Agent:** The "brain" — an LLM that decides what to do next.
- **Tools:** Functions the agent can call (e.g., search, calculator, database query).
- **Tool Call:** The agent's decision to use a specific tool with specific inputs.
- **Agent Executor:** The runtime that runs the agent loop (think → act → observe → repeat).

## 🖼️ Visual Diagram: The ReAct Loop

The agent's thinking process follows a pattern called **ReAct (Reason + Act)**.

```
            +----------------------------------------------------+
            |                                                    |
            |   +-----------+   +----------------------------+   |
User Input -> |    LLM      |-->| Thought: "I need to use a   |   |
            |  (Agent)    |   | tool. The 'Clock'     |   |
            |             |   | tool seems right."          |   |
            +-----------+   +----------------------------+   |
                  ^                                |         |
                  |                                v         |
            +-----------+   +----------------------------+   |
            | Tool Output |<--| Action: Use Clock          |   |
            | (e.g., 14:30)|   | with input (None).         |   |
            +-----------+   +----------------------------+   |
            |                                                    |
            +------------- Loop until final answer is found -----+
```

## ⚙️ Setup & Simple Example

Let's build our first agent! We'll give it a single tool: a clock.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import Tool
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Define a simple tool
def get_time(query: str = "") -> str:
    """Returns the current time in H:M format."""
    from datetime import datetime
    return datetime.now().strftime("%H:%M")

tools = [
    Tool(name="Clock", func=get_time, description="Returns the current time.")
]

# Create the prompt
# This is a simplified way to get a prompt that works with tool-calling models.
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# Create the agent
agent = create_tool_calling_agent(llm, tools, prompt)

# Create the agent executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

result = agent_executor.invoke({"input": "What time is it?"})

## 2. Built-in Agent Types

LangChain provides several pre-built agent types for common use cases. You can think of them as different 'personalities' or 'strategies' for the agent.

| Agent Type | Strategy | Best For |
|---|---|---|
| `zero-shot-react-description` | Simple, single-step tool use | Simple, direct tasks |
| `conversational-react-description` | Chat with memory | Building chatbots |
| `self-ask-with-search`| Breaks down questions into follow-up questions | Multi-hop questions (e.g., "Who was the president during the moon landing?") |
| `plan-and-execute` | Breaks a task into a plan, then executes steps | Complex, long-running workflows |
| `openai-tools` | Uses OpenAI's native tool-calling feature | GPT-3.5/4/4o (this is what we used above) |

The modern way to build agents is with `create_tool_calling_agent`, `create_react_agent`, or `create_self_ask_agent` which handle the prompt and model setup for you.

## 🔧 Hands-On Exercise

**Goal:** Create an agent that has two tools and can decide which one to use.

1.  Create a tool named `calculator` that takes a string expression (e.g., "2+2") and returns the result.
2.  Create a tool named `get_name_length` that takes a name and returns its length.
3.  Build an `AgentExecutor` with these two tools.
4.  Ask it a question that requires both tools, like: `"What is 5 times 4, and how many letters are in the name 'NVIDIA'?"`

In [ ]:
# 1. Calculator tool
def calculator(expression: str) -> str:
    """Evaluates a simple math expression."""
    return str(eval(expression))

# 2. Name length tool
def get_name_length(name: str) -> int:
    """Returns the number of letters in a name."""
    return len(name)

exercise_tools = [
    Tool(name="Calculator", func=calculator, description="A simple calculator."),
    Tool(name="NameLengthChecker", func=get_name_length, description="Checks the length of a name.")
]

# 3. Build the agent and executor
exercise_agent = create_tool_calling_agent(llm, exercise_tools, prompt)
exercise_executor = AgentExecutor(agent=exercise_agent, tools=exercise_tools, verbose=True)

# 4. Run it!
exercise_executor.invoke({"input": "What is 5 times 4, and how many letters are in the name 'NVIDIA'?"})

## 🐞 Bug Bounty

A common mistake is to define the tools for the agent but then pass a different set of tools to the `AgentExecutor`. The executor only knows about the tools you give it directly.

**Task:** The code below fails because the `AgentExecutor` doesn't have the `Clock` tool. Fix it by passing the correct `tools` list to the `AgentExecutor`.

In [ ]:
# --- BROKEN CODE ---
buggy_agent = create_tool_calling_agent(llm, tools, prompt)

# The agent knows about the Clock, but the executor doesn't!
buggy_executor = AgentExecutor(agent=buggy_agent, tools=[], verbose=True)

try:
    buggy_executor.invoke({"input": "What time is it?"})
except Exception as e:
    print(f"\nIt failed as expected! Error: {e}")

# --- FIXED CODE ---
print("\n--- Running fixed code ---")
fixed_executor = AgentExecutor(agent=buggy_agent, tools=tools, verbose=True)
fixed_executor.invoke({"input": "What time is it?"})

## 💡 Pro Tip

Always set `verbose=True` when you are developing or debugging an agent. Watching the agent's 'thought process' is the single most effective way to understand its behavior and fix problems. You can see exactly why it chose a tool, what input it used, and what it observed as the result.

## 🏁 Summary

You've taken your first step into the world of autonomous AI systems!

1.  **Agents Think and Act:** They use an LLM to reason about a task and decide which tools to use to solve it.
2.  **The `AgentExecutor` Runs the Loop:** This is the core component that drives the agent's `Reason -> Act` cycle.
3.  **Tool-Calling Models are the Future:** Modern LLMs have native tool-calling capabilities, making agents more reliable and easier to build than ever.